In [3]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input, Lambda, Conv2D
from tensorflow.keras.utils import register_keras_serializable
import time
import tkinter as tk
import tensorflow as tf
import random
import numpy as np
import gym2048
from tqdm import tqdm


model = Sequential([
    Input(shape=(4, 4, 12)),
    Flatten(),
    Dense(2048, activation='relu'),
    Dense(2048, activation='relu'),
    Dense(256, activation='relu'),
    Dense(4)  # One Q-value per action
])
model.compile(optimizer='adam', loss='mse')

In [4]:
num_episodes = 25000
epsilon_start = 0.9
epsilon_min = 0.01
decay_steps = 20000
target_update_interval = 250

from collections import deque
replay_buffer = deque(maxlen=50000)

batch_size = 64

optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)

target_model = tf.keras.models.clone_model(model)
target_model.compile(optimizer='adam', loss='mse')
target_model.set_weights(model.get_weights())


@tf.function
def train_step(states_one_hot, actions, rewards, next_states_one_hot, dones):

    huber = tf.keras.losses.Huber(
    delta=1.0,                                      # ⬅️ error clip
    reduction=tf.keras.losses.Reduction.NONE)         # ⬅️ no implicit mean

    with tf.GradientTape() as tape:
        q_values = model(states_one_hot, training=True)
        predicted_q = tf.reduce_sum(q_values * tf.one_hot(actions, 4), axis=1)

        next_q_values = target_model(next_states_one_hot, training=False)
        max_next_q = tf.reduce_max(tf.stop_gradient(next_q_values), axis=1)
        targets = rewards + 0.99 * max_next_q * (1.0 - tf.cast(dones, tf.float32))

        td_errors = tf.stop_gradient(targets - predicted_q)
        abs_td_err = tf.reduce_mean(tf.abs(td_errors))     # scalar

        loss_per_ex = huber(targets, predicted_q)
        loss = tf.reduce_mean(loss_per_ex)

        grads = tape.gradient(loss, model.trainable_weights)

        optimizer.apply_gradients(zip(grads, model.trainable_weights))
    return loss, abs_td_err

step = 0

for episode in tqdm(range(num_episodes)):
    state = gym2048.generate_board()
    done = False
    total_reward = 0

    while not done:

        step += 1
        epsilon = max(
            epsilon_min,
            epsilon_start - (epsilon_start - epsilon_min) * episode / decay_steps
        )

        if random.random() < epsilon:
            valid_actions = gym2048.get_valid_actions(state)
            action = random.choice(valid_actions)
        else:
            valid_actions = gym2048.get_valid_actions(state)

            state_tensor = tf.reshape(tf.one_hot(state, 12), (4, 4, 12))

            q_values = model(tf.expand_dims(state_tensor, axis=0), training=False).numpy()[0]

            invalid_actions = [a for a in range(4) if a not in valid_actions]
            q_values[invalid_actions] = -1e9

            action = int(np.argmax(q_values))  # guaranteed to be valid

        next_state, reward, done = gym2048.update_board(state, action)
        replay_buffer.append((state.copy(), action, reward, next_state.copy(), done))

        if step % 4 == 0 and len(replay_buffer) >= batch_size:
            batch = random.sample(replay_buffer, batch_size)
            states, actions, rewards, next_states, dones = map(np.array, zip(*batch))

            # Convert to float32 and int32 as appropriate
            states = tf.reshape(tf.one_hot(states, 12), (-1, 4, 4, 12))
            next_states = tf.reshape(tf.one_hot(next_states, 12), (-1, 4, 4, 12))
            rewards = tf.convert_to_tensor(rewards, dtype=tf.float32)
            dones = tf.convert_to_tensor(dones, dtype=tf.float32)
            actions = tf.convert_to_tensor(actions, dtype=tf.int32)

            loss, abs_td_error = train_step(states, actions, rewards, next_states, dones)

        state = next_state

        # Periodically update
        if step % target_update_interval == 0:
            target_model.set_weights(model.get_weights())

    if episode % 100 == 0:
        eval_score = 0
        for _ in range(5):
            #root = tk.Tk()
            board = gym2048.Game2048Board(None)
            #root.mainloop()
            d = False
            while not d:
                valid = gym2048.get_valid_actions(board.state)
                q = model(tf.expand_dims(tf.reshape(tf.one_hot(board.state, 12), (4, 4, 12)), axis=0), training=False)[0].numpy()
                q[[a for a in range(4) if a not in valid]] = -1e9
                a = np.argmax(q)
                s, r, d = gym2048.update_board(board.state, a)
                board.make_move(a)
                eval_score += r
            #root.destroy()
        print(f"Episode {episode} | Eval avg score: {eval_score / 5:.2f}")

  0%|          | 3/25000 [00:04<8:09:36,  1.18s/it] 

Episode 0 | Eval avg score: 466.34


  0%|          | 102/25000 [00:22<4:43:38,  1.46it/s]

Episode 100 | Eval avg score: 322.26


  1%|          | 201/25000 [00:40<4:58:23,  1.39it/s]

...

Episode 1100 | Eval avg score: 372.97


  5%|▍         | 1202/25000 [03:51<4:17:28,  1.54it/s]

Episode 1200 | Eval avg score: 226.49


  5%|▌         | 1300/25000 [04:13<1:16:58,  5.13it/s]


KeyboardInterrupt: 

In [ ]:
#board = gym2048.generate_board()

model.save("modelv1.h5")

# from tensorflow.keras.losses import MeanSquaredError

# model = tf.keras.models.load_model("modelv1.h5", compile=False)
# model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())



In [ ]:
import gym2048
import tkinter as tk
import numpy as np

import tensorflow as tf
model = tf.keras.models.load_model("modelv1.h5", compile=False)
model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())

done = False
board = gym2048.generate_board()

root = tk.Tk()
board = gym2048.Game2048Board(root)
board.let_model_play(model=model)
root.mainloop()



2025-06-13 17:48:16.636475: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-13 17:48:16.645401: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749829696.656060   17172 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749829696.659376   17172 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749829696.667663   17172 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

0
0
4
0
10.0
4
4
8
0
4
26.0
0
4
4
0
4
0
0
0
4
4
12
16
4
0
16
0
0
8
0
-0.1
-0.1
-0.1
-0.1
-0.1
-0.1
-0.1
-0.1
-0.1
-0.1
50.0
0
-0.1
-0.1
0
20
4
4
0
4
16
16
4
0
4
